In [23]:
import re
import pytesseract
from PIL import Image

In [24]:
# OCR the image
text = pytesseract.image_to_string(
    Image.open("../xraw-data/items-receipts/01.png"),
    config="--psm 6" #single uniform block of text
)
print(text)


sams club
CLUB MANAGER
ATLANTA, GA
01/17/25 18:30 1913 6643 90
0000323604 CENTER CUT 12.37 ¥
0000066846 CHICKEN 4.98 T
0000201898 BEEF STEAK 13.56 ¥
0000561914 MM WATER
4 AT 1 FOR 3.98 15.92 ¥
SUBTOTAL 46.83
TAX 1 8% 0.40
TAX 2 3% 1.26
TOTAL 48.49
VISA DEBIT TEND 48.49
VISA *#** KHER HHER 9943
CHANGE DUE 0.00
TC# 1732 0226 4261 5633 4989
01/17/25 18:30



In [25]:
# Find club manager
# Handles no club manager listed
club_manager = re.search(r'CLUB\s+MANAGER(?:[ ]([A-Z]+))?', text)

if club_manager:
    print(club_manager)
    print(club_manager.group(1))

<re.Match object; span=(10, 22), match='CLUB MANAGER'>
None


In [26]:
# Find date in format mm/dd/yy hh:mm
date = re.search(r'\d{2}/\d{2}/\d{2}\s+\d{2}:\d{2}', text)

if date:
    print(date)
    print("date group:", date.group(0))

<re.Match object; span=(35, 49), match='01/17/25 18:30'>
date group: 01/17/25 18:30


In [27]:
# Find location in format CITY, ST
# Handles multi-word city names 
location = re.search(r'[A-Z]+(?:[][A-Z]+)*,\s+[A-Z]{2}', text)

if location:
    print(location)
    print("location group:", location.group(0))

<re.Match object; span=(23, 34), match='ATLANTA, GA'>
location group: ATLANTA, GA


In [28]:
# Find items and prices including both single-line and double-line items
lines = text.splitlines()

# Process single-line items
# Handles items with weight and price on the same line
print("--single-line-items---\n")

for line in lines:
    single_line_item = re.match(r'^\d+\s+([A-Z][A-Z\s]+?)\s+(?:\d+\.\d{2}\s+)*(\d+\.\d{2})\s*.*$',line)
    
    if single_line_item:
        #print(single_line_item)
        print("item:", single_line_item.group(1))
        print("price:", single_line_item.group(2))

# Process double-line items
# Handles bulk items with name on first line and amount/price on second line
print("\n---double-line-items----\n")
pending_item = None

for line in lines:
    double_line_item = re.match(r'^\d+\s+([A-Z][A-Z\s]+)$',line)

    if double_line_item:
        #print(double_line_item)
        pending_item = double_line_item.group(1)
        print("item:", double_line_item.group(1))
        continue

    if pending_item:
        prices = re.match(r'^(\d+)\s+AT\s+\d+\s+FOR\s+(\d+\.\d{2})\s+(\d+\.\d{2})\s*.*$', line)
        print("quantity:", prices.group(1))
        print("unit price:", prices.group(2))
        print("total price:", prices.group(3))

        pending_item = None
    

--single-line-items---

item: CENTER CUT
price: 12.37
item: CHICKEN
price: 4.98
item: BEEF STEAK
price: 13.56

---double-line-items----

item: MM WATER
quantity: 4
unit price: 3.98
total price: 15.92


In [29]:
# Find subtotal in format SUBTOTAL 1234.56
subtotal = re.search(r'SUBTOTAL\s+(\d+\.\d{2})',text)

if subtotal:
    print(subtotal)
    print("subtotal group:", subtotal.group(1))

<re.Match object; span=(193, 207), match='SUBTOTAL 46.83'>
subtotal group: 46.83


In [30]:
# Find all taxes in format TAX 1 7% 12.34
# Returns list of tuples (tax number, tax rate, tax amount)
taxes = re.findall(r'TAX\s+(\d+)\s+(\d+%)\s+(\d+\.\d{2})',text)

if taxes:
    for tax in taxes:
        print("tax :", tax[0])
        print("tax rate:", tax[1])
        print("tax amount:", tax[2])
        print("-----")

tax : 1
tax rate: 8%
tax amount: 0.40
-----
tax : 2
tax rate: 3%
tax amount: 1.26
-----


In [31]:
# Find total in format TOTAL 1234.56
total = re.search(r'\bTOTAL\b\s+(\d+\.\d{2})', text)

if total:
    print(total)
    print("total group:", total.group(1))



<re.Match object; span=(236, 247), match='TOTAL 48.49'>
total group: 48.49


In [32]:
# Find payment method (VISA CREDIT)
paymentMethod = re.search(r'(VISA|MASTERCARD)\s+(CREDIT|DEBIT)',text)

if paymentMethod:
    print(paymentMethod)
    print("card:", paymentMethod.group(1))
    print("type:", paymentMethod.group(2))

<re.Match object; span=(248, 258), match='VISA DEBIT'>
card: VISA
type: DEBIT


In [33]:
# Find last 4 digits of card only (1234)
# Handles OCR junk between card type and digits

lastDigits = re.search(r'(?:VISA|MASTERCARD)[^\d]*(\d{4})', text)

if lastDigits:
    print(lastDigits)
    print("Last 4 digits:", lastDigits.group(1))

<re.Match object; span=(270, 294), match='VISA *#** KHER HHER 9943'>
Last 4 digits: 9943
